In [ ]:
import contextlib
from pathlib import Path
from uuid import uuid4

import polars as pl
import torch
import torch.nn.functional as F
from IPython.display import display
from lightning import Fabric

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.common.training import TrainingConfig
from src.config.base import BaseConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
)
from src.experiments.multimodal_gaussian.plotting.constraints import (
    plot_latents,
    plot_sample_pairs,
)

torch.set_float32_matmul_precision("medium")


In [2]:
num_modes = 4
mode_std = 0.1
offset = 1.0
ambient_dimension = 8

pretrain_chart_n_steps = 1_000
pretrain_critic_n_steps = 1_000
update_chart_every_n_critic_steps = 5
integrated_n_steps = 4_000
seed = 0
log_every_n_steps = 100
pretrain_plot_n_sample_pairs = 1_000
pretrain_plot_n_data_latents_per_mode = 1_000
boundary_eps = 1e-3

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

chart_transport_config = get_canonical_chart_transport_configs(
    data_config=data_config,
)

training_config = TrainingConfig.initialize(
    train_batch_size=2048,
    eval_batch_size=1024,
    eval_every_n_training_steps=3000,
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / uuid4().hex
    ),
)

chart_transport_config.visualize()


In [3]:
fabric = Fabric(
    accelerator="cuda", devices=1,
    precision="bf16-mixed",
)
fabric.launch()
fabric.seed_everything(seed)

device = fabric.device
device_type = device.type

runtime_data_config = data_config.replace(
    path="isometry",
    replacement=data_config.isometry.to(device=device, dtype=torch.float32),
).replace(
    path="projection",
    replacement=data_config.projection.to(device=device, dtype=torch.float32),
)

architecture_config = chart_transport_config.architecture_config
chart_transport_model = architecture_config.get_model()
optimizer = architecture_config.get_optimizer(
    model=chart_transport_model,
)
chart_transport_model, optimizer = fabric.setup(chart_transport_model, optimizer)

encoder = chart_transport_model.encoder
decoder = chart_transport_model.decoder
critic = chart_transport_model.critic

prior_config = chart_transport_config.prior_config
constraint_config = chart_transport_config.loss_config.constraint_config
constraint_method = constraint_config.constraint_method
pretrain_config = chart_transport_config.loss_config.chart_pretrain_config
transport_config = chart_transport_config.loss_config.transport_config

data_dual = torch.zeros((), device=device)
prior_dual = torch.zeros((), device=device)

fabric.print(
    f"device={device}, precision=bf16-mixed, folder={training_config.folder}"
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0


device=cuda:0, precision=bf16-mixed, folder=/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/fe64caef980a46e0aa7ab7b8aef9fe8a


In [ ]:
class PretrainMonitorConfig(BaseConfig):
    log_every_n_steps: int
    plot_n_sample_pairs: int
    plot_n_data_latents_per_mode: int

class CriticMonitorConfig(BaseConfig):
    """
    Config for visualizing the score implied by the critic. By one plot, we mean "html + png"
    1. Saves one plot for each sample_t_values, with two arrows attached to each point
        corresponding to the "data score" and the "prior - data score" at the noise level
    2. Saves one plot "transport "
        corresponding to the implied drift field for **clean data latent**,
        averaged across the noise spectrum. Background corresponds to contour lines
    3. Saves one plot "loss_spectrum" with x-axis being losses at
        unif[0, 1, interval=0.05] corresponding to the **score matching** loss at each point.
        Read the theory very carefully, I think this corresponds to noise-pred mse * (1-t)**2 / t ** 2
    Keep these specs after modification
    """
    log_every_n_steps: int
    sample_t_values: list[float] # Instantiate to [0.05, 0.2, 0.5, 0.7]. Save html-img for each
    num_contour_lines: int # Set to 10
    plot_n_data_latents_per_mode: int # Set to 1000
    plot_n_vectors_per_mode: int # Set to 50

class MonitorConfig(BaseConfig):
    pretrain_monitor_config: PretrainMonitorConfig

monitor_config = MonitorConfig(
    pretrain_monitor_config=PretrainMonitorConfig(
        log_every_n_steps=log_every_n_steps,
        plot_n_sample_pairs=pretrain_plot_n_sample_pairs,
        plot_n_data_latents_per_mode=pretrain_plot_n_data_latents_per_mode,
    )
)
monitor_config.visualize()

In [ ]:
def sample_monitor_batch(
    *,
    batch_size_per_mode: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batches = []
    labels = []
    for mode_id in range(runtime_data_config.num_modes):
        batches.append(
            runtime_data_config.sample_class(
                mode_id=mode_id,
                batch_size=batch_size_per_mode,
            )
        )
        labels.append(
            torch.full(
                (batch_size_per_mode,),
                fill_value=mode_id,
                device=device,
                dtype=torch.long,
            )
        )
    return torch.cat(batches, dim=0), torch.cat(labels, dim=0)


def sample_monitor_pairs_batch(
    *,
    total_batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    batch_size_per_mode = max(
        1,
        (total_batch_size + runtime_data_config.num_modes - 1)
        // runtime_data_config.num_modes,
    )
    samples, labels = sample_monitor_batch(
        batch_size_per_mode=batch_size_per_mode,
    )
    return samples[:total_batch_size], labels[:total_batch_size]


def write_plot_artifacts(
    *,
    figure,
    step: int,
    plot_type: str,
) -> None:
    output_folder = training_config.folder / "pretrain" / str(step)
    output_folder.mkdir(parents=True, exist_ok=True)
    figure.write_html(output_folder / f"{plot_type}.html")
    figure.write_image(output_folder / f"{plot_type}.png")


def monitor_pretrain_step(
    *,
    step: int,
) -> None:
    encoder_was_training = encoder.training
    decoder_was_training = decoder.training
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        pair_samples, pair_labels = sample_monitor_pairs_batch(
            total_batch_size=monitor_config.pretrain_monitor_config.plot_n_sample_pairs,
        )
        pair_latents = encoder(pair_samples)
        pair_reconstructions = decoder(pair_latents)
        pair_projected_samples, _, _ = data_config.decompose_projection(
            pair_samples.float().cpu(),
        )
        pair_projected_reconstructions, _, _ = data_config.decompose_projection(
            pair_reconstructions.float().cpu(),
        )
        pair_manifold_deviation = (
            (pair_reconstructions - pair_samples)
            .reshape(pair_samples.shape[0], -1)
            .norm(dim=-1)
            .float()
            .cpu()
        )

        latent_samples, latent_labels = sample_monitor_batch(
            batch_size_per_mode=monitor_config.pretrain_monitor_config.plot_n_data_latents_per_mode,
        )
        latent_values = encoder(latent_samples).float().cpu()
        _, _, latent_off_plane = data_config.decompose_projection(
            latent_samples.float().cpu(),
        )
        latent_off_manifold_norm = latent_off_plane.norm(dim=-1)

    if encoder_was_training:
        encoder.train()
    if decoder_was_training:
        decoder.train()

    sample_pairs_figure = plot_sample_pairs(
        samples=pair_projected_samples,
        pairs=pair_projected_reconstructions,
        manifold_deviation=pair_manifold_deviation,
        labels=pair_labels.cpu(),
    )
    write_plot_artifacts(
        figure=sample_pairs_figure,
        step=step,
        plot_type="sample_pairs",
    )

    latents_figure = plot_latents(
        latents=latent_values,
        off_manifold_norm=latent_off_manifold_norm,
        labels=latent_labels.cpu(),
    )
    write_plot_artifacts(
        figure=latents_figure,
        step=step,
        plot_type="latents",
    )


## Pretrain

In [5]:
pretrain_history = []
encoder.train()
decoder.train()
critic.eval()

for step in range(1, chart_transport_config.scheduling_config.pretrain_chart_n_steps + 1):
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        prior_batch = prior_config.sample(
            batch_size=training_config.train_batch_size,
        ).to(device=device, dtype=torch.float32)

        data_latents = encoder(data_batch)
        data_reconstruction = decoder(data_latents)
        prior_reconstruction = decoder(prior_batch)
        prior_latents = encoder(prior_reconstruction)

        data_cycle_loss = F.huber_loss(
            data_reconstruction,
            data_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        prior_cycle_loss = F.huber_loss(
            prior_latents,
            prior_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        constraint_loss = data_cycle_loss + prior_cycle_loss

        zero_mean_loss = data_latents.mean(dim=0).square().mean()
        latent_norms = data_latents.square().sum(dim=-1).sqrt()
        softplus_loss = F.softplus(
            latent_norms - pretrain_config.softplus_radius,
        ).mean()
        chart_loss = constraint_loss
        chart_loss = chart_loss + pretrain_config.zero_mean_weight * zero_mean_loss
        chart_loss = chart_loss + pretrain_config.softplus_weight * softplus_loss

    fabric.backward(chart_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "pretrain",
        "step": step,
        "chart_loss": chart_loss.detach().item(),
        "data_cycle_loss": data_cycle_loss.detach().item(),
        "prior_cycle_loss": prior_cycle_loss.detach().item(),
        "zero_mean_loss": zero_mean_loss.detach().item(),
        "softplus_loss": softplus_loss.detach().item(),
    }
    pretrain_history.append(metrics)

    if (
        step == 1
        or step % monitor_config.pretrain_monitor_config.log_every_n_steps == 0
        or step == chart_transport_config.scheduling_config.pretrain_chart_n_steps
    ):
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_chart_n_steps}: {summary}")
        monitor_pretrain_step(step=step)

pretrain_history = pl.DataFrame(pretrain_history)
display(pretrain_history.tail())


[pretrain] step 1/1000: chart_loss=0.6791, data_cycle_loss=0.1509, prior_cycle_loss=0.5276, zero_mean_loss=0.0505, softplus_loss=0.0154
[pretrain] step 100/1000: chart_loss=0.0060, data_cycle_loss=0.0028, prior_cycle_loss=0.0020, zero_mean_loss=0.0960, softplus_loss=0.0228
[pretrain] step 200/1000: chart_loss=0.0018, data_cycle_loss=0.0005, prior_cycle_loss=0.0006, zero_mean_loss=0.0533, softplus_loss=0.0203
[pretrain] step 300/1000: chart_loss=0.0013, data_cycle_loss=0.0002, prior_cycle_loss=0.0002, zero_mean_loss=0.0708, softplus_loss=0.0206
[pretrain] step 400/1000: chart_loss=0.0024, data_cycle_loss=0.0010, prior_cycle_loss=0.0007, zero_mean_loss=0.0511, softplus_loss=0.0192
[pretrain] step 500/1000: chart_loss=0.0021, data_cycle_loss=0.0008, prior_cycle_loss=0.0006, zero_mean_loss=0.0512, softplus_loss=0.0185
[pretrain] step 600/1000: chart_loss=0.0011, data_cycle_loss=0.0004, prior_cycle_loss=0.0002, zero_mean_loss=0.0378, softplus_loss=0.0179
[pretrain] step 700/1000: chart_loss

stage,step,chart_loss,data_cycle_loss,prior_cycle_loss,zero_mean_loss,softplus_loss
str,i64,f64,f64,f64,f64,f64
"""pretrain""",996,0.000535,0.000048,0.000055,0.025654,0.017528
"""pretrain""",997,0.000546,0.000052,0.000075,0.024418,0.01746
"""pretrain""",998,0.000501,0.000032,0.000052,0.024264,0.017455
"""pretrain""",999,0.000532,0.000052,0.000065,0.024027,0.017442
"""pretrain""",1000,0.000487,0.000041,0.000041,0.023156,0.017409


## Train noise critic

In [7]:
critic_pretrain_history = []
encoder.eval()
decoder.eval()
critic.train()

for step in range(1, chart_transport_config.scheduling_config.pretrain_critic_n_steps + 1):
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            data_latents = encoder(data_batch)

        t = boundary_eps + (1.0 - 2.0 * boundary_eps) * torch.rand(
            data_latents.shape[0],
            device=device,
        )
        eps = torch.randn_like(data_latents)
        noised_latents = (
            (1.0 - t).unsqueeze(-1) * data_latents + t.unsqueeze(-1) * eps
        )
        predicted_noise = critic(noised_latents, t)
        critic_loss = F.mse_loss(predicted_noise, eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "critic_pretrain",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
    }
    critic_pretrain_history.append(metrics)

    if step == 1 or step % log_every_n_steps == 0 or step == chart_transport_config.scheduling_config.pretrain_critic_n_steps:
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[critic_pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_critic_n_steps}: {summary}")

critic_pretrain_history = pl.DataFrame(critic_pretrain_history)
display(critic_pretrain_history.tail())


[critic_pretrain] step 1/2000: critic_loss=0.9031
[critic_pretrain] step 100/2000: critic_loss=0.2234
[critic_pretrain] step 200/2000: critic_loss=0.1839
[critic_pretrain] step 300/2000: critic_loss=0.1819
[critic_pretrain] step 400/2000: critic_loss=0.1843
[critic_pretrain] step 500/2000: critic_loss=0.1726
[critic_pretrain] step 600/2000: critic_loss=0.1633
[critic_pretrain] step 700/2000: critic_loss=0.1717
[critic_pretrain] step 800/2000: critic_loss=0.1764
[critic_pretrain] step 900/2000: critic_loss=0.1744
[critic_pretrain] step 1000/2000: critic_loss=0.1627
[critic_pretrain] step 1100/2000: critic_loss=0.1489
[critic_pretrain] step 1200/2000: critic_loss=0.1837
[critic_pretrain] step 1300/2000: critic_loss=0.1684
[critic_pretrain] step 1400/2000: critic_loss=0.1704
[critic_pretrain] step 1500/2000: critic_loss=0.1767
[critic_pretrain] step 1600/2000: critic_loss=0.1604
[critic_pretrain] step 1700/2000: critic_loss=0.1672
[critic_pretrain] step 1800/2000: critic_loss=0.1627
[crit

stage,step,critic_loss
str,i64,f64
"""critic_pretrain""",1996,0.167318
"""critic_pretrain""",1997,0.160627
"""critic_pretrain""",1998,0.161797
"""critic_pretrain""",1999,0.181185
"""critic_pretrain""",2000,0.165294


## Integrated training

In [8]:
integrated_history = []
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)

for step in range(1, integrated_n_steps + 1):
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        critic_data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        critic_t = boundary_eps + (1.0 - 2.0 * boundary_eps) * torch.rand(
            critic_data_latents.shape[0],
            device=device,
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = F.mse_loss(critic_predicted_noise, critic_eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        "chart_loss": float("nan"),
        "encoder_transport_loss": float("nan"),
        "decoder_transport_loss": float("nan"),
        "transport_field_norm": float("nan"),
        "data_cycle_loss": float("nan"),
        "prior_cycle_loss": float("nan"),
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }

    if step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with contextlib.ExitStack() as stack:
            if device_type == "cuda":
                stack.enter_context(torch.cuda.device(device))
            stack.enter_context(torch.device(str(device)))
            stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

            chart_data_batch = runtime_data_config.sample_unconditional(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transport_source_latents = encoder(chart_data_batch)
                time_bins = torch.arange(
                    transport_config.num_time_samples,
                    device=device,
                    dtype=torch.float32,
                )
                time_jitter = torch.rand(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    device=device,
                )
                transport_t = (time_bins.unsqueeze(0) + time_jitter) / transport_config.num_time_samples
                transport_t = transport_t.clamp(min=boundary_eps, max=1.0 - boundary_eps)

                transport_source_latents = transport_source_latents.unsqueeze(1).expand(
                    -1,
                    transport_config.num_time_samples,
                    -1,
                )
                transport_eps = torch.randn(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    transport_source_latents.shape[-1],
                    device=device,
                    dtype=transport_source_latents.dtype,
                )

                if transport_config.antipodal_estimate:
                    transport_t = torch.cat([transport_t, transport_t], dim=1)
                    transport_source_latents = transport_source_latents.repeat(1, 2, 1)
                    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

                transport_noised_latents = (
                    (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
                    + transport_t.unsqueeze(-1) * transport_eps
                )
                flat_transport_noised_latents = transport_noised_latents.reshape(
                    -1,
                    transport_noised_latents.shape[-1],
                )
                flat_transport_t = transport_t.reshape(-1)

                transport_predicted_noise = critic(
                    flat_transport_noised_latents,
                    flat_transport_t,
                ).reshape_as(transport_noised_latents)
                transport_prior_score = prior_config.analytic_score(
                    flat_transport_noised_latents.float(),
                    flat_transport_t.float(),
                ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
                    transport_noised_latents
                )
                transport_pullback_weight = (
                    transport_config.kl_weight_schedule.pullback_weight(
                        flat_transport_t.float(),
                    )
                    .to(dtype=flat_transport_noised_latents.dtype)
                    .reshape_as(transport_t)
                )
                transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
                    transport_prior_score
                    + transport_predicted_noise / transport_t.unsqueeze(-1)
                )

                if transport_config.antipodal_estimate:
                    midpoint = transport_config.num_time_samples
                    transport_field_terms = 0.5 * (
                        transport_field_terms[:, :midpoint]
                        + transport_field_terms[:, midpoint:]
                    )

                transport_field = transport_field_terms.mean(dim=1)
                transported_latents = (
                    encoder(chart_data_batch)
                    + transport_config.transport_step_size * transport_field
                )

            live_latents = encoder(chart_data_batch)
            encoder_transport_loss = F.huber_loss(
                live_latents,
                transported_latents,
                delta=transport_config.huber_delta,
                reduction="mean",
            )
            decoder_transport_loss = F.huber_loss(
                decoder(transported_latents),
                chart_data_batch,
                delta=transport_config.huber_delta,
                reduction="mean",
            )
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            data_dual = (
                data_dual
                + constraint_method.dual_variable_lr
                * (data_cycle_loss.detach() - constraint_method.data_constraint_budget)
            ).clamp_min(0.0)
            prior_dual = (
                prior_dual
                + constraint_method.dual_variable_lr
                * (prior_cycle_loss.detach() - constraint_method.prior_constraint_budget)
            ).clamp_min(0.0)

        metrics["chart_loss"] = chart_loss.detach().item()
        metrics["encoder_transport_loss"] = encoder_transport_loss.detach().item()
        metrics["decoder_transport_loss"] = decoder_transport_loss.detach().item()
        metrics["transport_field_norm"] = transport_field.norm(dim=-1).mean().detach().item()
        metrics["data_cycle_loss"] = data_cycle_loss.detach().item()
        metrics["prior_cycle_loss"] = prior_cycle_loss.detach().item()
        metrics["data_dual"] = data_dual.detach().item()
        metrics["prior_dual"] = prior_dual.detach().item()

    integrated_history.append(metrics)

    if step == 1 or step % log_every_n_steps == 0 or step == integrated_n_steps:
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[integrated] step {step}/{integrated_n_steps}: {summary}")

integrated_history = pl.DataFrame(integrated_history)
display(integrated_history.tail())


[integrated] step 1/4000: critic_loss=0.1576, chart_loss=0.0833, encoder_transport_loss=0.0333, decoder_transport_loss=0.0499, transport_field_norm=2.3915, data_cycle_loss=0.0000, prior_cycle_loss=0.0000, data_dual=0.0000, prior_dual=0.0000
[integrated] step 100/4000: critic_loss=0.5247, chart_loss=0.2406, encoder_transport_loss=0.1850, decoder_transport_loss=0.0365, transport_field_norm=4.0488, data_cycle_loss=0.0080, prior_cycle_loss=0.0111, data_dual=0.0016, prior_dual=0.0031
[integrated] step 200/4000: critic_loss=0.7094, chart_loss=0.3690, encoder_transport_loss=0.2906, decoder_transport_loss=0.0398, transport_field_norm=7.7314, data_cycle_loss=0.0150, prior_cycle_loss=0.0235, data_dual=0.0040, prior_dual=0.0092
[integrated] step 300/4000: critic_loss=0.5972, chart_loss=0.1981, encoder_transport_loss=0.1724, decoder_transport_loss=0.0166, transport_field_norm=4.8767, data_cycle_loss=0.0060, prior_cycle_loss=0.0031, data_dual=0.0047, prior_dual=0.0105
[integrated] step 400/4000: cr

stage,step,critic_loss,chart_loss,encoder_transport_loss,decoder_transport_loss,transport_field_norm,data_cycle_loss,prior_cycle_loss,data_dual,prior_dual
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""integrated""",3996,0.468332,0.116312,0.095019,0.010869,3.248471,0.002366,0.008234,0.005442,0.076104
"""integrated""",3997,0.490563,0.145896,0.119451,0.01434,3.815339,0.002535,0.009638,0.005434,0.076104
"""integrated""",3998,0.473801,0.141739,0.109509,0.015998,4.116457,0.00337,0.012693,0.005428,0.076106
"""integrated""",3999,0.493203,0.167052,0.131264,0.018947,4.685313,0.004261,0.012425,0.005422,0.076109
"""integrated""",4000,0.474768,0.16794,0.131163,0.018817,4.76459,0.00442,0.013317,0.005417,0.076112


In [9]:
integrated_history.hvplot.line(
    x="step",
    y=["chart_loss", "encoder_transport_loss", "decoder_transport_loss"]
)

:NdOverlay   [Variable]
   :Curve   [step]   (value)

In [10]:
integrated_history.hvplot.line(
    x="step",
    y=["data_cycle_loss", "prior_cycle_loss"]
)

:NdOverlay   [Variable]
   :Curve   [step]   (value)

In [11]:
integrated_history.hvplot.line(
    x="step",
    y=["transport_field_norm"]
)

:Curve   [step]   (transport_field_norm)